[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/purva-thakre/qce26-qrisp/blob/main/Session_2_Introduction_to_Jasp/JaspJIT.ipynb)

# JIT Compilation in JAX and Jasp

We know how to get a `jaxpr` out of a function (and a `jaspr` from a qrisp quantum program). 

In this notebook we see what `jax.jit` actually does with it: how it turns that jaxpr into compiled code that runs fast.

Most of the concepts in this notebook build on what we've already seen in the previous notebook. Here we just see the full pipeline instead of the first step (tracing).

## Setup

In [1]:
!pip install qrisp

In [2]:
import jax
import jax.numpy as jnp
import time

## Applying `jax.jit` for the first time

The most natural way to perform Just-In-Time compilation with JAX is through `jax.jit`. Let's see it in action.

We define a plain `f` Python function, which does some repeated array math.

In [3]:
def f(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)

At first, we time it as-is, so we have a plain baseline to compare every compiled version against later.

In [4]:
x = jnp.ones((1000, 1000))

start = time.perf_counter()
for _ in range(50):
    _ = f(x)
print(f"Without (jit): {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Without (jit): 123.72 ms per call


Let's see what changes once `jax.jit` gets involved.

`jax.jit(f)` doesn't compile anything by itself. It just wraps `f` so the *first* call traces and compiles it, and later calls (with matching input shapes/dtypes) **reuse** that compiled version.

Watch the timings below shift as we go from "no jit" to "first jit call" to "cached jit call."

In [5]:
f_jit = jax.jit(f)

Every timed call below ends in `.block_until_ready()`. This is necessary because JAX dispatches work asynchronously, so without it we'd be timing how fast Python hands off the call, not how long the computation actually takes (more on exactly why in the "Asynchronous Dispatch" section).

In [6]:
# First call (tracing + compilation)
start = time.perf_counter()
_ = f_jit(x).block_until_ready()
elapsed = (time.perf_counter() - start) * 1000
print(f"First call (trace + compile): {elapsed:.2f} ms")

First call (trace + compile): 130.92 ms


In [7]:
# Subsequent calls (use the cached compiled function)
start = time.perf_counter()
for _ in range(50):
    _ = f_jit(x).block_until_ready()
print(f"Subsequent calls: {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Subsequent calls: 3.79 ms per call


Observations:

- If we maintain the same shape and type of dynamic arguments, then the jaxpr is the same and, therefore, JAX does not have to re-compile our function: we have true just-in-time compilation!

- In practice, JAX doesn't retrace and compare jaxprs to decide this: it keys its cache on the input shapes/dtypes directly, so a matching signature skips tracing entirely and reuses the cached compiled version; a new signature forces a fresh trace (and, since the computation may now look different, possibly a different jaxpr).

- The first call already blends one real compilation together with 49 cache hits in the same timed loop. That's why its average doesn't look as expensive as a raw compile would.

We can also write the same function using `@jax.jit` as a decorator instead of calling `jax.jit(f)` directly:

In [8]:
@jax.jit
def f_decorated(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)


_ = f_decorated(x).block_until_ready()

## Asynchronous Dispatch and `block_until_ready` (bonus)

Calling a jitted function hands work off to the accelerator and returns right away. It doesn't block until the result is actually computed.

The first call below also doubles as a warm-up (so the timings that follow aren't paying a compilation cost); `block_until_ready()` is what forces Python to actually wait for a result.

In [9]:
@jax.jit
def heavy(a):
    return a @ a


a = jnp.ones((2000, 2000))
# this call triggers trace + compile.
# That is, a warm-up before we time dispatch vs. compute separately
heavy(a).block_until_ready()

Array([[2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       ...,
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.]], dtype=float32)

In [10]:
start = time.perf_counter()
result = heavy(a)
print(f"Call returned after: {(time.perf_counter() - start) * 1000:.3f} ms")

Call returned after: 0.292 ms


In [11]:
start = time.perf_counter()
heavy(a).block_until_ready()
print(f"Total time (dispatch + compute): {(time.perf_counter() - start) * 1000:.3f} ms")

Total time (dispatch + compute): 56.829 ms


Observations:

- The bare call above returns in a fraction of a millisecond. This is dispatch overhead, not the matmul itself: the accelerator keeps working after Python already moved on.

- Adding `block_until_ready()` exposes the real total time, dispatch plus compute together. This is the async gap in action.

## The Compilation Cache (bonus)

The jit cache is keyed on the *shape and dtype* of the inputs, not their concrete values. Same shape means a cache hit, a new shape means JAX has never seen this "version" of the function before.

This is exactly the same behaviour we have already seen for the `jaxpr`, which is conceptually the first IR of the entire Just-In-Time compilation process.
The difference now is that `jit` doesn't stop at the `jaxpr`: it keeps compiling all the way down to machine code. We open up exactly what happens in between later in this notebook, in "Inspecting What Gets Compiled".

The three calls below go compile → cache-hit → recompile, back to back.

In [12]:
@jax.jit
def g(x):
    return x * 2.0


# Compile for shape (10,)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Cache hit (same shape)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Recompile (new shape)
start = time.perf_counter()
_ = g(jnp.ones(20)).block_until_ready()
print(f"Recompile for shape (20,): {(time.perf_counter() - start) * 1000:.2f} ms")

Compiled for shape (10,): 43.46 ms
Compiled for shape (10,): 0.94 ms
Recompile for shape (20,): 78.01 ms


Same shape → fast (cache hit). 

new shape → slow again (fresh trace + compile), because the cache has no way to reuse a compiled program for an input signature it hasn't seen.

## Dynamic Loops in Jasp: `jrange`

As we have seen in the previous notebook, we cannot use a native Python `if` statement in case we have a predicate depending on a dynamic value, and the same happens for a loop: a plain Python `for i in range(n)` needs `n` to be a concrete Python integer at trace time, exactly like `if` needed a concrete `bool`.

`jrange` serves the same purpose as `jax.lax.fori_loop` from the previous notebook (a loop with a dynamic, traced bound), but it compiles directly to the underlying `while_loop` primitive that `fori_loop` itself dispatches to.

In [13]:
from qrisp import QuantumVariable, x, measure
from qrisp.jasp import jrange, make_jaspr


def test_f(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        x(qv[i])
    return measure(qv)


jaspr = make_jaspr(test_f)(1)
print(f"jaspr: {jaspr}")
print(f"jaspr(5): {jaspr(5)}")
print(f"jaspr(10): {jaspr(10)}")

jaspr: { lambda ; a:i64[] b:QuantumState. let
    c:QubitArray d:QuantumState = jasp.create_qubits a b
    e:i64[] = convert_element_type[new_dtype=int64 weak_type=False] a
    f:i64[] = sub e 1:i64[]
    g:i64[] = sub f f
    _:i64[] _:i64[] h:QuantumState = while[
      body_jaxpr={ lambda ; i:QubitArray j:i64[] k:i64[] l:QuantumState. let
          m:Qubit = jasp.get_qubit i k
          n:QuantumState = jasp.quantum_gate[gate=x] m l
          o:i64[] = add k 1:i64[]
          p:i64[] = jit[
            name=_jrange_marker
            jaxpr={ lambda ; o:i64[] j:i64[]. let  in (o,) }
          ] o j
        in (j, p, n) }
      body_nconsts=1
      cond_jaxpr={ lambda ; q:i64[] r:i64[] s:QuantumState. let
          t:bool[] = le r q
        in (t,) }
      cond_nconsts=0
    ] c f g d
    u:i64[] v:QuantumState = jasp.measure c h
  in (u, v) }
jaspr(5): 31                                                                    
jaspr(10): 1023                                               

Observations:

- `jrange(n)` looks exactly like a plain Python `for` loop, but `n` can be a traced value. Jasp records one loop body plus a runtime bound, compiled into a real `while_loop`, instead of unrolling.

- Calling the same jaspr with `n=5` and `n=10` doesn't retrace: it's just a different *value* for the bound, same as any other dynamic argument. Flipping all 5 qubits decodes to `31` ($2^5-1$); flipping all 10 decodes to `1023` ($2^{10}-1$).

- Unlike `jax.lax.fori_loop`, a `jrange` loop can't carry a value out to the code after it. Trying to return a running total from inside the loop raises an exception (values from the *previous* iteration are fine, only reaching past the loop's own boundary isn't).

## Resource Estimation at Scale

We've just seen `jax.jit`'s cache decide, based only on shape and dtype, whether to reuse a compiled program or trace again. Now let's see the first way Jasp gets to reuse that exact same compiled pipeline.

Jasp also ships decorators that compile a function into a *resource estimator* instead of a simulator or a real quantum computer.

For example, `count_ops` traces the function once and returns a dictionary of gate counts, without ever running an actual quantum simulation.

This works for exactly the reason we just saw with the shape/dtype-keyed `jit` cache above: if the size of a circuit is a *dynamic* argument (a traced value, not a Python `int` fixed ahead of time), tracing it once covers every possible size. Growing it doesn't trigger a new trace, only the classical gate-counting arithmetic has to scale.

In [14]:
from qrisp import count_ops, QuantumVariable, measure, h
from qrisp.jasp import jrange

# This is just a trick to count how many times the function is traced
trace_count = [0]


@count_ops(meas_behavior="0")
def main(n):
    trace_count[0] += 1
    qv = QuantumVariable(n)
    for i in jrange(n):
        h(qv[i])
    return measure(qv)


start = time.perf_counter()
result_small = main(5)
elapsed = (time.perf_counter() - start) * 1000
print(f"main(5): {elapsed:.1f} ms -> {result_small}")

start = time.perf_counter()
result_large = main(10_000_000)
elapsed = (time.perf_counter() - start) * 1000
print(
    f"main(10_000_000): {elapsed:.1f} ms -> total gates = {sum(result_large.values())}"
)

print(f"\nNumber of traces: {trace_count[0]}")

main(5): 68.6 ms -> {'h': 5, 'measure': 5}
main(10_000_000): 16.3 ms -> total gates = 20000000

Number of traces: 1


Observations:

- `main(5)` allocates a 5-qubit `QuantumVariable` and puts every qubit into superposition with `h`: `{'h': 5, 'measure': 5}`, exactly one Hadamard and one measurement per qubit.

- `main(10_000_000)` does the exact same thing on 10 million qubits (20 million gates total) and counting them still takes a few milliseconds.

- `trace_count` proves why: `n` is a dynamic argument, so growing it from 5 to 10,000,000 doesn't retrace at all. Only the classical gate-counting logic has to scale, not the number of times Python gets involved.

If you want to see this at scale, please check out [*Using `BigInteger` to compile Shor's algorithm at 2048 bit*](https://www.qrisp.eu/general/tutorial/BigInteger.html) in the official qrisp documentation. It applies this exact `count_ops` technique to resource-estimate Shor's algorithm at a cryptographically relevant size.

## Boolean Simulation

If a whole circuit is built only out of classical gates (that is, gates such as X, CX, CCX, SWAP, etc. which only ever permute basis states), there is no need for a real quantum-state simulator at all. That is, every qubit is either `0` or `1`, always, and every gate is just an ordinary bit operation.

`boolean_simulation` is a decorator defined within the Jasp submodule which exploits exactly this. It rewrites the Jaspr into a version with no quantum primitives left at all (qubits become classical bits packed into an array and gates become bitwise operations on that array). And because it's now ordinary JAX code, it goes through the real `jax.jit` pipeline we have seen above.

The catch: it only works for circuits that never leave classical territory. Anything that creates superposition (`h`, `rx`, `ry`, ...) will raise an exception.

In [15]:
from qrisp import QuantumVariable, measure, x
from qrisp.jasp import boolean_simulation, jrange


@boolean_simulation
def main(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        x(qv[i])
    return measure(qv)

In [16]:
from qrisp import count_ops


# Same circuit body, but count_ops (not boolean_simulation) to see its actual size
@count_ops(meas_behavior="0")
def main_gate_count(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        x(qv[i])
    return measure(qv)


gate_counts = main_gate_count(1000)
print(f"Total gates in this circuit: {sum(gate_counts.values())}")

# Initial run (compilation)
start = time.perf_counter()
result = main(1000)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

# Subsequent run (cache hit)
start = time.perf_counter()
result = main(1000)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

Total gates in this circuit: 2000
Result: -1, Time: 90.35 ms
Result: -1, Time: 0.52 ms


Observations:

- `main(1000)` flips every one of its 1000 qubits with `x`, deterministically: the result is identical on both calls, purely classical logic, purely classical result.

- This example has 2000 gates (1000 `x` + 1000 `measure`), and yet both calls finish in a fraction of a second. We can test examples at scale, at native JAX speed, with none of the exponential cost of a real quantum-state simulation.

## `qache`: Jasp's Compilation Cache (bonus)

This is the mechanism qrisp's Jasp module reuses for caching.

In Jasp, the `@qache` decorator allows us to mark a function as 'reusable' (that is, the jasp expression of this function will be cached and reused in the next calls if the function is called with the same signature).

Using the qache decorator not only improves the compilation speed but also enables the compiler to speed up transformation processes.

Below, we create a simple function that is 'qached'. 
To simulate an expensive compilation task we insert a `time.sleep` command:

In [17]:
from qrisp import h, cx, measure, QuantumVariable, QuantumFloat, make_jaspr
from qrisp.jasp import qache


@qache
def qached_function(qv):
    h(qv[0])
    cx(qv[0], qv[1])
    res_bl = measure(qv[0])
    # Simulate demanding compilation procedure by calling
    time.sleep(1)
    return res_bl


def main():
    a = QuantumVariable(2)
    b = QuantumFloat(2)
    bl_0 = qached_function(a)
    bl_1 = qached_function(b)
    bl_2 = qached_function(a)
    bl_3 = qached_function(b)

    return bl_0, bl_1, bl_2, bl_3


# Measure the time required for tracing
start = time.perf_counter()
jaspr = make_jaspr(main)()
print(f"Tracing time: {(time.perf_counter() - start) * 1000:.2f} ms")

Tracing time: 2018.50 ms


Even though `qached_function` has been called 4 times, we only see a delay of 2 seconds. 

This is because the function has been called with two different quantum types, implying it has been traced twice and recalled from the cache twice. 

We can see all of this by looking at the `jaspr`:

In [18]:
print(jaspr)

let qached_function = { lambda ; a:QubitArray b:QuantumState. let
    c:Qubit = jasp.get_qubit a 0:i64[]
    d:QuantumState = jasp.quantum_gate[gate=h] c b
    e:Qubit = jasp.get_qubit a 1:i64[]
    f:QuantumState = jasp.quantum_gate[gate=cx] c e d
    g:bool[] h:QuantumState = jasp.measure c f
  in (g, h) } in
let qached_function1 = { lambda ; i:QubitArray j:i64[] k:QuantumState. let
    l:Qubit = jasp.get_qubit i 0:i64[]
    m:QuantumState = jasp.quantum_gate[gate=h] l k
    n:Qubit = jasp.get_qubit i 1:i64[]
    o:QuantumState = jasp.quantum_gate[gate=cx] l n m
    p:bool[] q:QuantumState = jasp.measure l o
  in (p, q) } in
{ lambda ; r:QuantumState. let
    s:QubitArray t:QuantumState = jasp.create_qubits 2:i64[] r
    u:QubitArray v:QuantumState = jasp.create_qubits 2:i64[] t
    w:bool[] x:QuantumState = jit[name=qached_function jaxpr=qached_function] s v
    y:bool[] z:QuantumState = jit[name=qached_function jaxpr=qached_function1] u
      0:i64[] x
    ba:bool[] bb:QuantumState

We see three different function definitions in the `jaspr`:

- The first one describes `qached_function` called with a `QuantumVariable`. For this kind of signature only the `QubitArray` is required.

- The second one describes `qached_function` called with `QuantumFloat`. Additionally to the `QubitArray`, the `.exponent` attribute is also passed as a traced value (the extra `i64[]` argument).

- The third function definition is `main`, which calls the previously defined functions.

Notice that a `jaspr` can be simulated using the built-in real-time simulator. We achieve this by calling the jaspr like a function:

In [19]:
# Notice that `time.sleep` is not measured
# in the execution time as it is a side effect
jaspr()

(True, False, True, False)

This is qrisp's own default simulator. That is, not the pipeline `boolean_simulation` used earlier in this notebook. 

Calling a jaspr this way still walks every quantum primitive through a custom Python interpreter, one equation at a time (never through `jit` → StableHLO → XLA). `boolean_simulation` only reaches that pipeline because it removes every quantum primitive first. By default, Jasp does not.

### Exercise (bonus): Predicting a Jaspr

Based on what we have seen up to this point, we can try to predict some properties of a `jaspr`.

Let's take a small, fixed quantum function:

In [20]:
from qrisp import QuantumVariable, x, cx, h, measure


def simple_circuit():
    qv = QuantumVariable(3)
    x(qv[0])
    cx(qv[0], qv[1])
    h(qv[2])
    return measure(qv)

Before running anything, how many equations do you expect the resulting `jaspr` to contain? 

Recall that allocating the register, fetching each qubit, and applying each gate are all separate equations.

In [21]:
# Your code here: build the jaspr with make_jaspr and print it
...

Ellipsis

Before running this next part, what do you expect when you call the jaspr as a function (simulating it) several times in a row? 

Will the result always be the same, or can it vary? Think about which of the three gates above behaves differently from the other two.

In [22]:
# Your code here: call the jaspr as a function a few times and observe
...

Ellipsis

### Solution

In [23]:
from qrisp import make_jaspr

jaspr = make_jaspr(simple_circuit)()
print(jaspr)

{ lambda ; a:QuantumState. let
    b:QubitArray c:QuantumState = jasp.create_qubits 3:i64[] a
    d:Qubit = jasp.get_qubit b 0:i64[]
    e:QuantumState = jasp.quantum_gate[gate=x] d c
    f:Qubit = jasp.get_qubit b 1:i64[]
    g:QuantumState = jasp.quantum_gate[gate=cx] d f e
    h:Qubit = jasp.get_qubit b 2:i64[]
    i:QuantumState = jasp.quantum_gate[gate=h] h g
    j:i64[] k:QuantumState = jasp.measure b i
  in (j, k) }


Observations:

- **8 equations**: `create_qubits` once, then `get_qubit` + `quantum_gate` for each of the 3 gates (6 equations), then `measure` once. `1 + 6 + 1 = 8`.

- Notice `cx(qv[0], qv[1])` only triggers *one* new `get_qubit` (for `qv[1]`): `qv[0]`'s qubit (variable `d`) was already fetched for the `x` gate right before, and gets reused directly as the control input, instead of being fetched again.

In [24]:
for _ in range(6):
    print(jaspr())

3                                                                               
7                                                                               
3                                                                               
7                                                                               
7                                                                               
3                                                                               


Observations:

- The result alternates between `3` (`0b011`) and `7` (`0b111`), never anything else: qubit 0 is always `1` (`x`), qubit 1 is always `1` (`cx`, since qubit 0 is always `1`), so those two bits are fully deterministic.

- Only qubit 2 varies, because `h` is the one gate here that creates superposition. Everything else in this circuit (`x`, `cx`) only ever permutes basis states, exactly like the classical gates from `boolean_simulation` earlier in this notebook, just simulated here on a real statevector.

## Inspecting What Gets Compiled (bonus)

`jit` builds a compiled program behind the scenes, and `.lower(x)` lets you grab a handle on it without running it, and print it out at different levels of representation.

We have to make a distinction here: StableHLO is a portable, versioned dialect meant to be exchanged across tools and JAX/XLA versions (hence "stable"), while HLO is XLA's own internal representation, free to change under the hood. Below is the same function shown as both.

In [25]:
@jax.jit
def h(x):
    return jnp.sin(x) + jnp.cos(x)

In [26]:
x = jnp.ones(4)

In [27]:
lowered = h.lower(x)
print(lowered.as_text(dialect="stablehlo"))

module @jit_h attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4xf64>) -> (tensor<4xf64> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<4xf64>
    %1 = stablehlo.cosine %arg0 : tensor<4xf64>
    %2 = stablehlo.add %0, %1 : tensor<4xf64>
    return %2 : tensor<4xf64>
  }
}



In [28]:
print(lowered.as_text(dialect="hlo"))

HloModule jit_h, entry_computation_layout={(f64[4]{0})->f64[4]{0}}

ENTRY main.5 {
  x.1 = f64[4]{0} parameter(0)
  sin.2 = f64[4]{0} sine(x.1)
  cos.3 = f64[4]{0} cosine(x.1)
  ROOT add.4 = f64[4]{0} add(sin.2, cos.3)
}




Same three operations either way. Notice HLO is basically the StableHLO version respelled in XLA's native syntax, not a different computation.

## `jit` Itself Is a Primitive (bonus)

As we saw with jaxprs, every JAX operation becomes a primitive in the trace, and `jit` is no exception.

When a jitted function is called from inside another traced function, it isn't inlined. It shows up as its own `jit` primitive wrapping a nested jaxpr.

In [ ]:
def outer(arg):
    @jax.jit
    def inner(x):
        return x + arg * jnp.ones(1)

    return arg + inner(arg - 2.0)

In [30]:
print(jax.make_jaxpr(outer)(1.0))

{ lambda ; a:f64[]. let
    b:f64[] = sub a 2.0:f64[]
    c:f64[1] = jit[
      name=inner
      jaxpr={ lambda ; a:f64[] b:f64[]. let
          d:f64[1] = broadcast_in_dim[
            broadcast_dimensions=()
            shape=(1,)
            sharding=None
          ] 1.0:f64[]
          e:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
          f:f64[1] = mul e d
          g:f64[] = convert_element_type[new_dtype=float64 weak_type=False] b
          c:f64[1] = add g f
        in (c,) }
    ] a b
    h:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
    i:f64[1] = add h c
  in (i,) }


See the `jit[name=inner, jaxpr={...}]` entry in the middle of the printout: that's `inner` kept as its own nested sub-program rather than flattened into `outer`'s jaxpr.

## `disable_jit` for Debugging (bonus)

`jax.disable_jit()` turns every `jit` inside the block into a no-op, so code runs as plain, eager Python: handy for stepping through with a debugger or trusting every `print` you see.

Compare how many times "Tracing!" prints with and without it:

In [ ]:
@jax.jit
def f(x):
    print("Tracing!")
    return x * 2.0

In [32]:
print("=== With jit ===")
_ = f(jnp.array(1.0))
_ = f(jnp.array(2.0))

=== With jit ===
Tracing!


In [33]:
print("=== With disable_jit ===")
with jax.disable_jit():
    _ = f(jnp.array(1.0))
    _ = f(jnp.array(2.0))

=== With disable_jit ===
Tracing!
Tracing!


Observations:

- With `jit`: tracing (and the print) happens once. The second call is a cache hit and stays silent.

- With `disable_jit`: there's no cache at all, so every call re-runs the Python body from scratch, printing every time.

## Integration with Catalyst

So far every jaxpr and jaspr in this notebook has run through either JAX's own `jit`/XLA path or Jasp's own Python-level interpreter. There's a third option: compiling all the way down to a real runtime via [Catalyst](https://docs.pennylane.ai/projects/catalyst/), which is Xanadu's own JIT/AOT compiler for JAX and PennyLane programs.

Catalyst is an optional extra, not installed by `qrisp` by default: `pip install qrisp[catalyst]`.

In [34]:
!pip install qrisp[catalyst]

/home/pietrop/anaconda3/envs/Qrisp_3_11/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


qrisp's `qjit` bridges *into* Catalyst by tracing our function to a `jaspr`, translating every Jasp primitive into Catalyst's own equivalent (`quantum_gate` → Catalyst's `qinst`, `measure` → Catalyst's own `measure`, and so on), then constructing and running an actual `catalyst.QJIT` instance. This is the exact same class a standalone PennyLane script would get from `@catalyst.qjit`. Just like `sin` lowers into `stablehlo.sine`, `qinst` itself lowers further into the `quantum.custom` ops shown below.

Its calling convention mirrors `jax.jit` exactly. The first call traces and compiles (now all the way to a runnable artifact), a later call with a matching signature is cached:

In [35]:
from qrisp import QuantumFloat, cx, h, measure
from qrisp.jasp import qjit


@qjit
def test_fun(i):
    qv = QuantumFloat(i)
    cx(qv[0], qv[1])
    h(qv[1])
    return measure(qv)


start = time.perf_counter()
result_first = test_fun(4)
elapsed = (time.perf_counter() - start) * 1000
print(f"First call (compile + run): {elapsed:.1f} ms -> {result_first}")

start = time.perf_counter()
result_cached = test_fun(4)
print(f"Cached call: {(time.perf_counter() - start) * 1000:.1f} ms -> {result_cached}")

First call (compile + run): 671.2 ms -> 2.0
Cached call: 37.7 ms -> 0.0


Observations:

- This has the same shape as every other cache-hit demo in this notebook: a slow first call (trace + compile + run), a fast cached call. Just with a full hardware-compilation backend behind it instead of XLA.

- The two calls return different numbers even for the same `i=4`: `test_fun` genuinely measures a quantum state partway through, so its result is a real random variable, not a deterministic function of `i`.

We can see the translation directly: `.to_catalyst_mlir()` on a `jaspr` returns Catalyst's own MLIR dialect.

In [36]:
from qrisp import QuantumFloat, cx, h, measure
from qrisp.jasp import make_jaspr


def example_function(i):
    qv = QuantumFloat(i)
    cx(qv[0], qv[1])
    h(qv[1])
    meas_res = measure(qv)
    return meas_res + 1


jaspr = make_jaspr(example_function)(2)
print(jaspr.to_catalyst_mlir())

module @jaspr_function {
  func.func public @jit_jaspr_function(%arg0: tensor<i64>) -> tensor<f64> attributes {llvm.emit_c_interface} {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f64>
    %c = stablehlo.constant dense<1024> : tensor<i64>
    %c_0 = stablehlo.constant dense<false> : tensor<i1>
    %c_1 = stablehlo.constant dense<1> : tensor<i64>
    %c_2 = stablehlo.constant dense<30> : tensor<i64>
    %c_3 = stablehlo.constant dense<[30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]> : tensor<30xi64>
    %c_4 = stablehlo.constant dense<0> : tensor<i64>
    %c0_i64 = arith.constant 0 : i64
    quantum.device shots(%c0_i64) ["/home/pietrop/anaconda3/envs/Qrisp_3_11/lib/python3.11/site-packages/pennylane_lightning/liblightning_qubit_catalyst.so", "LightningSimulator", "{'mcmc': False, 'num_burnin': 0, 'kernel_name': None}"] {auto_qubit_management}
    %0 = quantum.alloc( 20) : !quantum.reg
    %1 = stablehlo.broa

Observations:

- `quantum.device`, `quantum.alloc`, `quantum.custom "cx"` and so on are Catalyst's own dialect. This is a different spellings entirely from the `jasp.*` primitives used everywhere else in this notebook.

- Classical structure doesn't get reinvented for Catalyst: the `stablehlo.constant`/`stablehlo.while` machinery is still plain JAX StableHLO. Only the quantum-specific parts get translated into Catalyst's vocabulary.

The `jaspr` object has also the `to_qir()` method, which goes one level lower than the MLIR dialect above: real QIR (which is just LLVM IR with a specific set of quantum conventions).

Under the hood, both `to_catalyst_mlir()` and `to_qir()` both go through the exact same internal Catalyst compilation call, which builds a `catalyst.QJIT` object, generates its MLIR, then compiles it further. 

The `to_catalyst_mlir()` method hands us the MLIR stage, while `to_qir()` hands us what that same compilation produces afterward.

In [ ]:
qir_text = jaspr.to_qir()
print(f"Total length: {len(qir_text)} characters\n")
print(qir_text[:216])

Total length: 10827 characters

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"


Observations:

- Genuine LLVM IR syntax (`ModuleID`, `source_filename`, `target datalayout`, `target triple`): nothing left of Catalyst's own `quantum.*` vocabulary at this stage, and nothing of `jasp.*` either. This is the actual compilation target, not a dialect anymore.

- Same `jaspr`, same underlying Catalyst compilation as the MLIR example above, just a later stage of it. That's also why it's roughly 10x longer: MLIR is still fairly high-level, LLVM IR spells out every low-level detail.

## Keep Exploring

We've only had time to cover `qache`, `boolean_simulation`, and a first look at resource estimation and tracing. Jasp has quite a bit more: repeat-until-success loops, real-time classical control flow mixed with quantum operations (even running neural network inference inside a coherence window), and more.

For the full picture, see the [Jasp documentation](https://www.qrisp.eu/general/tutorial/Jasp.html). Hopefully this tutorial gave you a solid enough high-level understanding to keep exploring from here on your own.

## ... what comes next?

If you want to go further, the `BonusMaterial/` folder has three optional notebooks: 

- `PrimitivesInJax.ipynb` (build a custom JAX primitive from scratch, with rules for implementation, abstract evaluation, lowering, etc.),

- `XLABackend.ipynb` (how XLA actually compiles and fuses a jitted program), and 

- `StableHLOExport.ipynb` (inspecting compiled cost/memory, and exporting a compiled artifact).